# Imbalanced Data

**Author:** Tohidul Islam Tareq

Handling class imbalance with resampling and class weights.

In [8]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (7, 4)


In [9]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, matthews_corrcoef
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

X, y = make_classification(n_samples=1200, n_features=2, n_redundant=0, n_informative=2,
                           n_clusters_per_class=1, weights=[0.90, 0.10], class_sep=1.2,
                           random_state=RANDOM_STATE)
print('Original distribution:', Counter(y))
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.30, random_state=RANDOM_STATE)


Original distribution: Counter({np.int64(0): 1074, np.int64(1): 126})


In [10]:
strategies = {
    'baseline': None,
    'class_weight_balanced': 'class_weight',
    'random_oversampling': RandomOverSampler(random_state=RANDOM_STATE),
    'smote': SMOTE(random_state=RANDOM_STATE),
    'random_undersampling': RandomUnderSampler(random_state=RANDOM_STATE)
}

rows = []
for name, strategy in strategies.items():
    if strategy == 'class_weight':
        X_fit, y_fit = X_train, y_train
        model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
    elif strategy is None:
        X_fit, y_fit = X_train, y_train
        model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    else:
        X_fit, y_fit = strategy.fit_resample(X_train, y_train)
        model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    model.fit(X_fit, y_fit)
    pred = model.predict(X_test)
    rows.append({'strategy': name, 'mcc': matthews_corrcoef(y_test, pred), 'resampled_distribution': str(Counter(y_fit))})
pd.DataFrame(rows).sort_values('mcc', ascending=False)


,strategy,mcc,resampled_distribution
0,baseline,0.833217,"Counter({np.int64(0): 752, np.int64(1): 88})"
3,smote,0.762135,"Counter({np.int64(0): 752, np.int64(1): 752})"
1,class_weight_balanced,0.715962,"Counter({np.int64(0): 752, np.int64(1): 88})"
2,random_oversampling,0.715962,"Counter({np.int64(0): 752, np.int64(1): 752})"
4,random_undersampling,0.683332,"Counter({np.int64(0): 88, np.int64(1): 88})"
